### Imports

In [ ]:
import json
import os
from dataclasses import asdict
from pathlib import Path
from typing import List, Callable

from objectives import evaluate_all_objectives, evaluate_diagnosis_objective
from preprocess import (
    PatientGroundTruth,
    extract_tool_use,
    load_one_result,
    match_ground_truth_and_assistant,
    merge_called_args,
)
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import logging
import warnings

from dotenv import load_dotenv
from openai import OpenAI
from fastapi.encoders import jsonable_encoder

from dataset.mimic_dataset import MIMIC_Dataset

import sys

sys.path.append("..")
from config import DATASET_NAMES
from paths import EVALUABLE_OUTPUTS_BASELINE_DIR, EVALUABLE_OUTPUTS_BIAS_DIR, EVALUATIONS_DIR

#### Variables

Set up logging, client and paths

In [ ]:
# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ignore warnings
warnings.filterwarnings("ignore")

load_dotenv()
client = OpenAI()

DEFAULT_INPUT_DIR = str(EVALUABLE_OUTPUTS_BASELINE_DIR)
DEFAULT_OUTPUT_DIR = str(EVALUATIONS_DIR / "MIRA_EVALUATED_OUTPUTS")
RUN_PARALLEL = True


In [ ]:
def get_file_paths(base_path: str) -> List[Path]:
    """
    Recursively get all .jsonl file paths under the given base path.

    Args:
        base_path (str): The base directory to search for files.

    Returns:
        List[Path]: A list of file paths.
    """
    return list(Path(base_path).rglob("*.jsonl"))


def evaluate_one_file(
    client: OpenAI,
    file_path: Path,
    dataset: MIMIC_Dataset,
    out_dir: str,
    diagnosis: str,
    diagnosis_name_for_evaluation: str,
    evaluate_function: Callable[[OpenAI, dict, str], dict],
) -> None:
    """
    Evaluate a single assistant output file against the ground truth and write the evaluation to the output directory.

    Args:
        client (OpenAI): The OpenAI client.
        file_path (Path): The path to the assistant output file.
        dataset (MIMIC_Dataset): The dataset object containing patient data.
        out_dir (str): The base output directory.
        diagnosis (str): The diagnosis for the path creation.
        diagnosis_name_for_evaluation (str): The diagnosis_name_for_evaluation subdirectory name. For use as evaluation criterion in the evaluation.
        evaluate_function (Callable[[OpenAI, dict, str], dict]): The function to evaluate the diagnosis.
    """
    try:
        output_dir = Path(out_dir) / diagnosis
        output_dir.mkdir(parents=True, exist_ok=True)
        output_file_path = output_dir / f"{Path(file_path).stem}.json"

        if os.path.exists(output_file_path):
            logger.info(f"Skipping file {file_path} because it already exists")
            return

        med_assistant_outputs, patient_data, metadata = load_one_result(
            file_path, dataset
        )

        # Process ground truth and assistant outputs
        patient_gt = PatientGroundTruth(patient_data)
        med_assistant_tools_used = extract_tool_use(med_assistant_outputs)
        med_called_args = merge_called_args(med_assistant_tools_used)
        matched_results = match_ground_truth_and_assistant(med_called_args, patient_gt)

        # Evaluate objectives
        evaluation = evaluate_function(
            client, matched_results, diagnosis_name_for_evaluation
        )

        # Prepare output directory
        # Write evaluation results to JSON
        evaluation_dict = asdict(evaluation)

        evaluation_result = {
            "evaluation": evaluation_dict,
            "matched_results": jsonable_encoder(matched_results),
            "metadata": metadata,
        }

        evaluation_json = json.dumps(evaluation_result, indent=2)

        with open(output_file_path, "w") as output_file:
            output_file.write(evaluation_json)

        # try:
        #     with open(output_file_path, "w") as output_file:
        #         output_file.write(evaluation_json)

        # except PermissionError:
        #     logger.info(
        #         f"File {output_file_path} already exists. Do you want to force overwrite?"
        #     )
        #     overwrite = input("Enter 'y' to overwrite, 'n' to skip: ")
        #     if overwrite.lower() != "y":
        #         logger.info(f"Skipping file {output_file_path}")
        #         return
        #     else:
        #         os.chmod(
        #             output_file_path, 0o666
        #         )  # Change file permissions to allow overwriting
        #         with open(output_file_path, "w") as output_file:
        #             output_file.write(evaluation_json)

        # os.chmod(output_file_path, 0o444)

        logger.info(
            f"Successfully evaluated and saved results for {Path(output_file_path).name}"
        )
    except Exception as e:
        logger.error(f"Error processing file {file_path}: {e}", exc_info=True)


def evaluate_file_wrapper(
    client,
    file_path,
    dataset,
    out_dir,
    diagnosis,
    diagnosis_name_for_evaluation,
    evaluate_function,
):
    evaluate_one_file(
        client=client,
        file_path=file_path,
        dataset=dataset,
        out_dir=out_dir,
        diagnosis=diagnosis,
        diagnosis_name_for_evaluation=diagnosis_name_for_evaluation,
        evaluate_function=evaluate_function,
    )

In [ ]:
DIAGNOSIS_MAP = {
    "appendicitis": "appendicitis",
    "cholecystitis": "cholecystitis",
    "diverticulitis": "diverticulitis",
    "pancreatic_cancer": "pancreatic cancer",
    "pancreatitis": "pancreatitis",
    "pneumonia": "pneumonia",
    "pulmonary_embolism": "pulmonary embolism",
    "uti": "urinary tract infection",
}

for diagnosis in DATASET_NAMES:
    dataset = MIMIC_Dataset.load_dataset(diagnosis)
    file_paths = get_file_paths(os.path.join(DEFAULT_INPUT_DIR, diagnosis))

    diagnosis_name_for_evaluation = DIAGNOSIS_MAP[diagnosis]

    if RUN_PARALLEL:
        with ThreadPoolExecutor(max_workers=40) as executor:
            list(
                tqdm(
                    executor.map(
                        lambda file_path: evaluate_file_wrapper(
                            client,
                            file_path,
                            dataset,
                            DEFAULT_OUTPUT_DIR,
                            diagnosis,
                            diagnosis_name_for_evaluation,
                            evaluate_diagnosis_objective,
                        ),
                        file_paths,
                    ),
                    desc=f"Evaluating output files for {diagnosis}",
                    total=len(file_paths),
                )
            )
    else:
        for file_path in tqdm(file_paths, desc=f"Evaluating output files for {diagnosis}"):
            evaluate_one_file(
                client,
                file_path,
                dataset,
                DEFAULT_OUTPUT_DIR,
                diagnosis,
                diagnosis_name_for_evaluation,
                evaluate_diagnosis_objective,
            )

#### Run all objectives

In [ ]:
DIAGNOSIS_MAP = {
    "appendicitis": "appendicitis",
    "cholecystitis": "cholecystitis",
    "diverticulitis": "diverticulitis",
    "pancreatic_cancer": "pancreatic cancer",
    "pancreatitis": "pancreatitis",
    "pneumonia": "pneumonia",
    "pulmonary_embolism": "pulmonary embolism",
    "uti": "urinary tract infection",
}

for diagnosis in DATASET_NAMES:
    dataset = MIMIC_Dataset.load_dataset(diagnosis)
    file_paths = get_file_paths(os.path.join(DEFAULT_INPUT_DIR, diagnosis))

    diagnosis_name_for_evaluation = DIAGNOSIS_MAP[diagnosis]

    if RUN_PARALLEL:
        with ThreadPoolExecutor(max_workers=10) as executor:
            list(
                tqdm(
                    executor.map(
                        lambda file_path: evaluate_file_wrapper(
                            client,
                            file_path,
                            dataset,
                            DEFAULT_OUTPUT_DIR,
                            diagnosis,
                            diagnosis_name_for_evaluation,
                            evaluate_all_objectives,
                        ),
                        file_paths,
                    ),
                    desc=f"Evaluating output files for {diagnosis}",
                    total=len(file_paths),
                )
            )
    else:
        for file_path in tqdm(file_paths, desc=f"Evaluating output files for {diagnosis}"):
            evaluate_one_file(
                client,
                file_path,
                dataset,
                DEFAULT_OUTPUT_DIR,
                diagnosis,
                diagnosis_name_for_evaluation,
                evaluate_all_objectives,
            )

### Evaluate Bias

In [ ]:
# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ignore warnings
warnings.filterwarnings("ignore")

load_dotenv()
client = OpenAI()

biases = ("recency_bias", "healthy_bias", "anxiety_bias", "french_bias", "german_bias", "sex")

DEFAULT_DIRS = {
    str(EVALUABLE_OUTPUTS_BIAS_DIR / f"EVALUABLE_OUTPUTS_BIAS_{bias}"): str(EVALUATIONS_DIR / f"MIRA_EVALUATED_OUTPUTS_{bias}")
    for bias in biases
}

RUN_PARALLEL = True


In [ ]:
DIAGNOSIS_MAP = {
    "appendicitis": "appendicitis",
    "cholecystitis": "cholecystitis",
    "diverticulitis": "diverticulitis",
    "pancreatic_cancer": "pancreatic_cancer",
    "pancreatitis": "pancreatitis",
    "pneumonia": "pneumonia",
    "pulmonary_embolism": "pulmonary embolism",
    "uti": "urinary tract infection",
}

for DEFAULT_INPUT_DIR, DEFAULT_OUTPUT_DIR in DEFAULT_DIRS.items():
    for diagnosis in DATASET_NAMES:
        dataset = MIMIC_Dataset.load_dataset(diagnosis)
        file_paths = get_file_paths(os.path.join(DEFAULT_INPUT_DIR, diagnosis))

        diagnosis_name_for_evaluation = DIAGNOSIS_MAP.get(diagnosis)
        if diagnosis_name_for_evaluation is None:
            continue

        else:
            print(f"Running for {diagnosis}")
            if RUN_PARALLEL:
                with ThreadPoolExecutor(max_workers=10) as executor:
                    list(
                        tqdm(
                            executor.map(
                                lambda file_path: evaluate_file_wrapper(
                                    client,
                                    file_path,
                                    dataset,
                                    DEFAULT_OUTPUT_DIR,
                                    diagnosis,
                                    diagnosis_name_for_evaluation,
                                    evaluate_all_objectives,
                                ),
                                file_paths,
                            ),
                            desc=f"Evaluating output files for {diagnosis}",
                            total=len(file_paths),
                        )
                    )
            else:
                for file_path in tqdm(file_paths, desc=f"Evaluating output files for {diagnosis}"):
                    evaluate_one_file(
                        client,
                        file_path,
                        dataset,
                        DEFAULT_OUTPUT_DIR,
                        diagnosis,
                        diagnosis_name_for_evaluation,
                        evaluate_all_objectives,
                    )

            print(DEFAULT_OUTPUT_DIR)